In [1]:
import requests
import pandas as pd
import json
import time

In [2]:
BANK_NAME = "SC"
BANK_CODE = "SC"

SC_URL = "https://www.standardchartered.co.kr/np/kr/prdctList"
REFERER_URL = "https://www.standardchartered.co.kr/np/kr/pl/et/InterestRateDeposit4P.jsp?utm_source=chatgpt.com"

COMMON_HEADERS = {
    "accept": "application/json, text/javascript, */*; q=0.01",
    "content-type": "application/json;charset=UTF-8",
    "origin": "https://www.standardchartered.co.kr",
    "referer": REFERER_URL,
    "user-agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/146.0.0.0 Safari/537.36"
    ),
    "x-requested-with": "XMLHttpRequest",
}

In [3]:
def generate_month_end_dates(start_date="2004-01-31", end_date="2019-12-31"):
    dates = []
    current = pd.to_datetime(start_date) + pd.offsets.MonthEnd(0)
    end = pd.to_datetime(end_date)

    while current <= end:
        dates.append(current.strftime("%Y%m%d"))
        current = current + pd.offsets.MonthEnd(1)

    return dates

In [4]:
def build_sc_payload(target_yyyymmdd):
    payload = {
        "serviceID": "HP_AP_RM_FxRate.selectSearchRateList",
        "SCFB_MESSAGE_ID": "HP_DpH718I01_H718_IN",
        "IMS_TRAN_CODE": "TI1IBF01",
        "IN_CLASS_CODE": "H718",
        "JOB_TYPE": "GZ",
        "SERVICE_CODE": "718",
        "TSPassword": "111111",
        "UserID": "FIRST900",
        "referDate": target_yyyymmdd
    }
    return payload

In [5]:
def fetch_sc_json(session, target_yyyymmdd, debug=False):
    payload = build_sc_payload(target_yyyymmdd)

    resp = session.post(
        SC_URL,
        headers=COMMON_HEADERS,
        json=payload,
        timeout=60
    )
    resp.raise_for_status()

    if debug:
        print("status:", resp.status_code)
        print("content-type:", resp.headers.get("content-type"))
        print(resp.text[:1000])

    return resp.json()

In [6]:
def normalize_sc_rows(raw_json, target_yyyymmdd):
    columns = [
        "bank", "bank_code", "target_date",
        "currency", "maturity", "rate", "product"
    ]

    try:
        vector = raw_json["HP_DpH718I01_H718_OUT"]["ARR"]["vector"]
    except Exception:
        return pd.DataFrame(columns=columns)

    target_date_fmt = pd.to_datetime(
        target_yyyymmdd, format="%Y%m%d"
    ).strftime("%Y-%m-%d")

    maturity_map = {
        "normalAmt": "보통예금",
        "noticeAmt": "통지예금",
        "onevWeak": "1주",
        "oneMonth": "1개월",
        "twoMonth": "2개월",
        "threeMonth": "3개월",
        "fourMonth": "4개월",
        "fiveMonth": "5개월",
        "sixMonth": "6개월",
    }

    rows = []

    for item in vector:
        inner = item.get("HP_DpH718I01_H718_OUT_ARR", {})
        currency = inner.get("currName")
        adobe_gb = inner.get("adobeGb")

        # 사용자 친화적으로 표시
        if adobe_gb == "1":
            resident_label = "거주자"
        elif adobe_gb == "2":
            resident_label = "비거주자"
        else:
            resident_label = f"구분{adobe_gb}"

        for raw_key, maturity_label in maturity_map.items():
            raw_rate = inner.get(raw_key)

            rows.append({
                "bank": BANK_NAME,
                "bank_code": BANK_CODE,
                "target_date": target_date_fmt,
                "currency": currency,
                "maturity": maturity_label,
                "rate": pd.to_numeric(raw_rate, errors="coerce") / 1000,
                "product": f"외화예금 ({resident_label})"
            })

    df = pd.DataFrame(rows).drop_duplicates().reset_index(drop=True)
    return df

In [7]:
session = requests.Session()
test_date = "20260330"

raw_json = fetch_sc_json(session, test_date, debug=True)
df_test = normalize_sc_rows(raw_json, test_date)

print("rows:", len(df_test))
print(df_test.head(20))

status: 200
content-type: application/json;charset=UTF-8
{"HP_DpH718I01_H718_OUT":{"ARR":{"vector":[{"HP_DpH718I01_H718_OUT_ARR":{"oneMonth":"03067","adobeGb":"1","sixMonth":"03345","fourMonth":"03300","fiveMonth":"03353","normalAmt":"00020","threeMonth":"03237","onevWeak":"02778","noticeAmt":"02778","currName":"USD","twoMonth":"03152"}},{"HP_DpH718I01_H718_OUT_ARR":{"oneMonth":"00165","adobeGb":"1","sixMonth":"00521","fourMonth":"00477","fiveMonth":"00499","normalAmt":"00000","threeMonth":"00457","onevWeak":"00000","noticeAmt":"00000","currName":"JPY","twoMonth":"00311"}},{"HP_DpH718I01_H718_OUT_ARR":{"oneMonth":"03445","adobeGb":"1","sixMonth":"03651","fourMonth":"03484","fiveMonth":"03592","normalAmt":"01031","threeMonth":"03553","onevWeak":"02815","noticeAmt":"02815","currName":"GBP","twoMonth":"03504"}},{"HP_DpH718I01_H718_OUT_ARR":{"oneMonth":"01489","adobeGb":"1","sixMonth":"01803","fourMonth":"01714","fiveMonth":"01803","normalAmt":"00515","threeMonth":"01714","onevWeak":"01360

In [8]:
print(type(raw_json))
print(raw_json.keys())

out = raw_json.get("HP_DpH718I01_H718_OUT", {})
print(out.keys())

arr = out.get("ARR", {})
print(arr.keys())

vector = arr.get("vector", [])
print("vector length:", len(vector))

if len(vector) > 0:
    print(vector[0].keys())
    print(vector[0]["HP_DpH718I01_H718_OUT_ARR"])

<class 'dict'>
dict_keys(['HP_DpH718I01_H718_OUT'])
dict_keys(['ARR', 'ScfbHeader', 'rtrvDate', 'rtrvTime', 'rcvDetailCount', 'UserID', 'rtrvBasicDate', 'userId'])
dict_keys(['vector'])
vector length: 38
dict_keys(['HP_DpH718I01_H718_OUT_ARR'])
{'oneMonth': '03067', 'adobeGb': '1', 'sixMonth': '03345', 'fourMonth': '03300', 'fiveMonth': '03353', 'normalAmt': '00020', 'threeMonth': '03237', 'onevWeak': '02778', 'noticeAmt': '02778', 'currName': 'USD', 'twoMonth': '03152'}


In [9]:
def crawl_sc_range(start_date="2004-01-31", end_date="2019-12-31", sleep_sec=0.3):
    session = requests.Session()
    all_frames = []
    failed_dates = []

    date_list = generate_month_end_dates(start_date, end_date)
    print(f"총 {len(date_list)}개 날짜 수집 시작")

    for i, yyyymmdd in enumerate(date_list, 1):
        try:
            raw_json = fetch_sc_json(session, yyyymmdd, debug=False)
            df_day = normalize_sc_rows(raw_json, yyyymmdd)

            if not df_day.empty:
                all_frames.append(df_day)

            print(f"[{i}/{len(date_list)}] {yyyymmdd} 완료 - {len(df_day)} rows")

        except Exception as e:
            print(f"[{i}/{len(date_list)}] {yyyymmdd} 실패 - {e}")
            failed_dates.append({
                "target_date": yyyymmdd,
                "error": str(e)
            })

        time.sleep(sleep_sec)

    if all_frames:
        final_df = pd.concat(all_frames, ignore_index=True)
        final_df = final_df.drop_duplicates().reset_index(drop=True)
    else:
        final_df = pd.DataFrame(columns=[
            "bank", "bank_code", "target_date",
            "currency", "maturity", "rate", "product"
        ])

    failed_df = pd.DataFrame(failed_dates)
    return final_df, failed_df

In [10]:
sc_df, failed_df = crawl_sc_range(
    start_date="2004-01-31",
    end_date="2019-12-31",
    sleep_sec=0.3
)

print("\n최종 shape:", sc_df.shape)
print(sc_df.head())
print("\n실패 건수:", len(failed_df))
print(failed_df.head())

총 192개 날짜 수집 시작
[1/192] 20040131 실패 - Expecting value: line 1 column 1 (char 0)
[2/192] 20040229 실패 - Expecting value: line 1 column 1 (char 0)
[3/192] 20040331 완료 - 324 rows
[4/192] 20040430 완료 - 324 rows
[5/192] 20040531 완료 - 324 rows
[6/192] 20040630 완료 - 324 rows
[7/192] 20040731 실패 - Expecting value: line 1 column 1 (char 0)
[8/192] 20040831 완료 - 324 rows
[9/192] 20040930 완료 - 324 rows
[10/192] 20041031 실패 - Expecting value: line 1 column 1 (char 0)
[11/192] 20041130 완료 - 324 rows
[12/192] 20041231 완료 - 324 rows
[13/192] 20050131 완료 - 324 rows
[14/192] 20050228 완료 - 324 rows
[15/192] 20050331 완료 - 324 rows
[16/192] 20050430 실패 - Expecting value: line 1 column 1 (char 0)
[17/192] 20050531 완료 - 324 rows
[18/192] 20050630 완료 - 324 rows
[19/192] 20050731 실패 - Expecting value: line 1 column 1 (char 0)
[20/192] 20050831 완료 - 324 rows
[21/192] 20050930 완료 - 324 rows
[22/192] 20051031 완료 - 324 rows
[23/192] 20051130 완료 - 324 rows
[24/192] 20051231 실패 - Expecting value: line 1 column 1 (ch

In [11]:
print("고유 날짜 수:", sc_df["target_date"].nunique())

print("\n날짜별 행 수:")
print(sc_df["target_date"].value_counts().head())

print("\n통화별 행 수:")
print(sc_df["currency"].value_counts().head(20))

print("\n상품별 행 수:")
print(sc_df["product"].value_counts())

print("\n만기별 행 수:")
print(sc_df["maturity"].value_counts())

print("\n샘플:")
print(sc_df.sample(min(20, len(sc_df)), random_state=42))

고유 날짜 수: 135

날짜별 행 수:
target_date
2010-08-31    342
2010-09-30    342
2010-11-30    342
2010-12-31    342
2011-01-31    342
Name: count, dtype: int64

통화별 행 수:
currency
USD    2430
JPY    2430
GBP    2430
CAD    2430
CHF    2430
HKD    2430
SEK    2430
AUD    2430
DKK    2430
NOK    2430
SAR    2430
KWD    2430
AED    2430
SGD    2430
MYR    2430
NZD    2430
THB    2430
EUR    2430
CNY    1440
Name: count, dtype: int64

상품별 행 수:
product
외화예금 (거주자)     22590
외화예금 (비거주자)    22590
Name: count, dtype: int64

만기별 행 수:
maturity
보통예금    5020
통지예금    5020
1주      5020
1개월     5020
2개월     5020
3개월     5020
4개월     5020
5개월     5020
6개월     5020
Name: count, dtype: int64

샘플:
      bank bank_code target_date currency maturity   rate      product
19458   SC        SC  2011-01-31      KWD     보통예금  0.161  외화예금 (비거주자)
34472   SC        SC  2016-05-31      AUD       1주  1.871  외화예금 (비거주자)
5457    SC        SC  2005-11-30      AED      1개월  4.257  외화예금 (비거주자)
40942   SC        SC  2018-07-31      C

In [12]:
output_file = "sc_2004_2019_full.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    sc_df.to_excel(writer, sheet_name="Sheet1", index=False)
    failed_df.to_excel(writer, sheet_name="failed_dates", index=False)

print(f"저장 완료: {output_file}")

저장 완료: sc_2004_2019_full.xlsx
